# League of Legends - Previsao de Vitoria em Ranked
## Script Mestre - Gera o Projeto Completo (7 Fases)

**Como usar:**
1. Execute Celula 1 (instala dependencias)
2. Execute Celula 2 (upload do CSV)
3. Execute as demais celulas em ordem
4. No final, baixe o ZIP com tudo organizado

---

In [1]:
# CELULA 1 - Instalar dependencias
import subprocess
result = subprocess.run(['pip', 'install', 'xgboost', 'pyarrow', '-q'], capture_output=True)
print('Dependencias instaladas.')

Dependencias instaladas.


In [2]:
# CELULA 2 - Setup de pastas e upload do CSV
import os, sqlite3, warnings, json, joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import gaussian_kde
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, roc_curve, silhouette_score, silhouette_samples)
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

BASE_DIR = '/content/lol_project'
DIRS = {
    'notebooks': os.path.join(BASE_DIR, 'notebooks'),
    'figures':   os.path.join(BASE_DIR, 'outputs', 'figures'),
    'data':      os.path.join(BASE_DIR, 'outputs', 'data'),
    'models':    os.path.join(BASE_DIR, 'outputs', 'models'),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print('Pastas criadas.')
from google.colab import files
print('Selecione o arquivo Base_M43_Pratique_LOL_RANKED_WIN.csv:')
uploaded = files.upload()
nome = list(uploaded.keys())[0]
CSV_PATH = os.path.join(BASE_DIR, 'Base_M43_Pratique_LOL_RANKED_WIN.csv')
with open(CSV_PATH, 'wb') as f:
    f.write(uploaded[nome])
print('CSV salvo:', CSV_PATH)

Pastas criadas.
Selecione o arquivo Base_M43_Pratique_LOL_RANKED_WIN.csv:


Saving Base_M43_Pratique_LOL_RANKED_WIN.csv to Base_M43_Pratique_LOL_RANKED_WIN.csv
CSV salvo: /content/lol_project/Base_M43_Pratique_LOL_RANKED_WIN.csv


In [3]:
# CELULA 3 - FASE 1: SQL e DataFrame
print('FASE 1 - Setup, SQLite e Ingestao')
df_raw = pd.read_csv(CSV_PATH)
print('CSV carregado:', df_raw.shape)

conn = sqlite3.connect(os.path.join(BASE_DIR, 'lol_ranked.db'))
df_raw.to_sql('lol_matches', conn, if_exists='replace', index=False, chunksize=1000)
count = conn.execute('SELECT COUNT(*) FROM lol_matches').fetchone()[0]
print('SQLite populado:', count, 'registros')

sql = (
    "WITH base AS ("
    "SELECT *, "
    "CAST(blueKills AS FLOAT)/NULLIF(blueKills+blueDeaths,0) AS blue_kda_ratio, "
    "blueWardsPlaced - redWardsDestroyed AS blue_vision_advantage, "
    "blueKills - redKills AS kills_diff, "
    "blueEliteMonsters - redEliteMonsters AS elite_monster_diff, "
    "blueTotalMinionsKilled + blueTotalJungleMinionsKilled AS blue_total_cs, "
    "redTotalMinionsKilled + redTotalJungleMinionsKilled AS red_total_cs, "
    "CASE WHEN blueGoldDiff > 1500 THEN 'Forte Vantagem Azul' "
    "     WHEN blueGoldDiff > 0    THEN 'Leve Vantagem Azul' "
    "     WHEN blueGoldDiff = 0    THEN 'Equilibrio' "
    "     WHEN blueGoldDiff > -1500 THEN 'Leve Desvantagem Azul' "
    "     ELSE 'Forte Desvantagem Azul' END AS categoria_gold_diff "
    "FROM lol_matches) "
    "SELECT * FROM base ORDER BY gameId"
)
df = pd.read_sql_query(sql, conn)
conn.close()
df.to_parquet(os.path.join(DIRS['data'], 'df_fase1.parquet'), index=False)
print('DataFrame criado:', df.shape)
print('FASE 1 concluida!')

FASE 1 - Setup, SQLite e Ingestao
CSV carregado: (9879, 40)
SQLite populado: 9879 registros
DataFrame criado: (9879, 47)
FASE 1 concluida!


In [4]:
# CELULA 4 - Configuracao de estilo global
BLUE   = '#2563EB'; RED  = '#DC2626'; GOLD  = '#D97706'; GREEN  = '#16A34A'
PURPLE = '#7C3AED'; ORANGE = '#EA580C'; TEAL = '#0D9488'
GRAY   = '#6B7280'; BG = '#F8FAFC'; WHITE = '#FFFFFF'
CLUSTER_COLORS = [BLUE, RED, GREEN, GOLD]
FIGS = DIRS['figures']

plt.rcParams.update({
    'figure.facecolor':BG, 'axes.facecolor':WHITE,
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.color':'#E5E7EB', 'grid.linewidth':0.6,
    'font.family':'DejaVu Sans', 'axes.titlesize':13, 'axes.labelsize':11,
    'xtick.labelsize':9, 'ytick.labelsize':9, 'legend.frameon':False,
})
print('Estilo configurado.')

Estilo configurado.


In [5]:
# CELULA 5 - FASE 2: Graficos EDA 01 a 04
print('Gerando graficos da Fase 2...')
vc = df['blueWins'].value_counts()

# GRAFICO 01
fig = plt.figure(figsize=(16, 10), facecolor=BG)
fig.suptitle('Visao Geral do Dataset', fontsize=16, fontweight='bold', color='#1E293B', y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

ax1 = fig.add_subplot(gs[0, 0])
wedges, _ = ax1.pie([vc[1], vc[0]], colors=[BLUE, RED], startangle=90,
                     wedgeprops=dict(width=0.55, edgecolor=WHITE, linewidth=2.5))
ax1.text(0, 0, str(len(df)) + chr(10) + 'partidas', ha='center', va='center',
         fontsize=11, fontweight='bold', color='#1E293B')
ax1.legend(wedges, ['Vitoria Azul ' + str(vc[1]), 'Derrota Azul ' + str(vc[0])],
           loc='lower center', bbox_to_anchor=(0.5, -0.22), fontsize=9)
ax1.set_title('Distribuicao do Target', fontweight='bold')

ax2 = fig.add_subplot(gs[0, 1])
dv = df[df['blueWins']==1]['blueKills']
dd_v = df[df['blueWins']==0]['blueKills']
bp = ax2.boxplot([dv, dd_v], patch_artist=True, widths=0.45,
                  medianprops=dict(color=WHITE, linewidth=2.5),
                  flierprops=dict(marker='o', markersize=3, alpha=0.3))
bp['boxes'][0].set_facecolor(BLUE); bp['boxes'][1].set_facecolor(RED)
for w in bp['whiskers'] + bp['caps']: w.set_color(GRAY)
ax2.set_xticks([1, 2]); ax2.set_xticklabels(['Vitoria', 'Derrota'])
ax2.set_ylabel('Kills'); ax2.set_title('Kills por Resultado', fontweight='bold')

ax3 = fig.add_subplot(gs[0, 2])
fb_wr  = df[df['blueFirstBlood']==1]['blueWins'].mean() * 100
nfb_wr = df[df['blueFirstBlood']==0]['blueWins'].mean() * 100
bars3  = ax3.bar(['Com FB', 'Sem FB'], [fb_wr, nfb_wr], color=[BLUE, RED],
                  width=0.5, edgecolor=WHITE, linewidth=1.5)
ax3.set_ylim(0, 80); ax3.axhline(50, color=GRAY, linestyle='--', linewidth=1, alpha=0.7)
ax3.set_ylabel('Taxa de Vitoria (%)'); ax3.set_title('Impacto do First Blood', fontweight='bold')
for bar, val in zip(bars3, [fb_wr, nfb_wr]):
    ax3.text(bar.get_x()+bar.get_width()/2, val+1.5,
             str(round(val,1))+'%', ha='center', fontweight='bold', fontsize=12)

ax4 = fig.add_subplot(gs[1, :2])
wg = df[df['blueWins']==1]['blueGoldDiff']
lg = df[df['blueWins']==0]['blueGoldDiff']
ax4.hist(wg, bins=60, alpha=0.65, color=BLUE, label='Vitoria', density=True)
ax4.hist(lg, bins=60, alpha=0.65, color=RED,  label='Derrota', density=True)
ax4.axvline(wg.mean(), color=BLUE, linestyle='--', linewidth=2, label='Media Vitoria: +'+str(round(wg.mean(),0)))
ax4.axvline(lg.mean(), color=RED,  linestyle='--', linewidth=2, label='Media Derrota: '+str(round(lg.mean(),0)))
ax4.set_xlabel('Gold Diff'); ax4.set_ylabel('Densidade')
ax4.set_title('Distribuicao da Diferenca de Ouro', fontweight='bold'); ax4.legend()

ax5 = fig.add_subplot(gs[1, 2]); ax5.axis('off')
cols_d = ['blueKills', 'blueTotalGold', 'blueGoldDiff', 'blueGoldPerMin']
lbl_d  = ['Kills', 'TotalGold', 'GoldDiff', 'Gold/Min']
desc = df[cols_d].describe().loc[['mean','std','min','50%','max']].round(1)
desc.index = ['Media','DesvP','Min','Mediana','Max']; desc.columns = lbl_d
tbl = ax5.table(cellText=desc.values, rowLabels=desc.index, colLabels=desc.columns,
                cellLoc='center', loc='center', bbox=[0, 0.05, 1, 0.9])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#E5E7EB')
    if r == 0: cell.set_facecolor(BLUE); cell.set_text_props(color=WHITE, fontweight='bold')
    elif c == -1: cell.set_facecolor('#F1F5F9'); cell.set_text_props(fontweight='bold')
    else: cell.set_facecolor(WHITE)
ax5.set_title('Estatistica Descritiva', fontweight='bold')

plt.savefig(os.path.join(FIGS, 'grafico_01_visao_geral.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_01 salvo')

# GRAFICO 02
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('O Ouro Decide a Partida', fontsize=16, fontweight='bold', y=1.02)

ax = axes[0]
bins2 = pd.qcut(df['blueGoldDiff'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
wr_q  = df.groupby(bins2, observed=True)['blueWins'].mean() * 100
brs   = ax.bar(wr_q.index, wr_q.values, color=[RED,'#F97316','#84CC16',GREEN], width=0.55, edgecolor=WHITE, linewidth=2)
ax.axhline(50, color=GRAY, linestyle='--', linewidth=1.5); ax.set_ylim(0, 95)
ax.set_ylabel('Taxa de Vitoria (%)'); ax.set_title('Win Rate por Quartil de Gold Diff', fontweight='bold')
for b, v in zip(brs, wr_q.values):
    ax.text(b.get_x()+b.get_width()/2, v+1, str(round(v,1))+'%', ha='center', fontweight='bold', fontsize=12)

ax = axes[1]
smp = df.sample(1500, random_state=42)
sw  = smp[smp['blueWins']==1]; sl = smp[smp['blueWins']==0]
ax.scatter(sl['blueGoldDiff'], sl['kills_diff'], color=RED,  alpha=0.35, s=18, label='Derrota')
ax.scatter(sw['blueGoldDiff'], sw['kills_diff'], color=BLUE, alpha=0.35, s=18, label='Vitoria')
ax.axvline(0, color='#374151', linewidth=1.5, linestyle=':', alpha=0.7)
ax.axhline(0, color='#374151', linewidth=1.5, linestyle=':', alpha=0.7)
ax.set_xlabel('Gold Diff'); ax.set_ylabel('Kills Diff')
ax.set_title('Gold Diff x Kills Diff', fontweight='bold'); ax.legend()

ax = axes[2]
df['dragon_cat'] = pd.cut(df['blueDragons'], bins=[-1,0,1,2,5],
                           labels=['0 Dragoes','1 Dragao','2 Dragoes','3+'])
pivot = df.groupby(['categoria_gold_diff','dragon_cat'], observed=True)['blueWins'].mean().unstack() * 100
ordem = ['Forte Desvantagem Azul','Leve Desvantagem Azul','Leve Vantagem Azul','Forte Vantagem Azul']
pivot = pivot.reindex([o for o in ordem if o in pivot.index])
sns.heatmap(pivot, ax=ax, cmap='RdYlGn', annot=True, fmt='.0f',
            linewidths=0.5, vmin=0, vmax=100, annot_kws={'size':10,'weight':'bold'})
ax.set_title('Win Rate: Gold x Dragoes', fontweight='bold')
ax.tick_params(axis='x', rotation=0); ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'grafico_02_ouro_decide.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_02 salvo')

# GRAFICO 03
fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor=BG)
fig.suptitle('Os Primeiros 10 Minutos Definem o Jogo', fontsize=16, fontweight='bold', y=1.01)

ax = axes[0, 0]
parts = ax.violinplot([df[df['blueWins']==1]['blueGoldPerMin'].values,
                       df[df['blueWins']==0]['blueGoldPerMin'].values],
                      positions=[1, 2], showmedians=True)
for pc, col in zip(parts['bodies'], [BLUE, RED]):
    pc.set_facecolor(col); pc.set_alpha(0.65)
parts['cmedians'].set_color(WHITE); parts['cmedians'].set_linewidth(2.5)
for p in ['cbars','cmins','cmaxes']: parts[p].set_color(GRAY)
ax.set_xticks([1, 2]); ax.set_xticklabels(['Vitoria','Derrota'])
ax.set_ylabel('Gold por Minuto'); ax.set_title('Gold/Min por Resultado', fontweight='bold')

ax = axes[0, 1]
oc = ['blueFirstBlood','blueDragons','blueHeralds','blueTowersDestroyed']
ol = ['First Blood','Dragoes','Arautos','Torres']
mv2 = df[df['blueWins']==1][oc].mean()
md2 = df[df['blueWins']==0][oc].mean()
xp2 = np.arange(len(ol)); w2 = 0.35
b1 = ax.bar(xp2-w2/2, mv2, w2, label='Vitoria', color=BLUE, edgecolor=WHITE, linewidth=1.5)
b2 = ax.bar(xp2+w2/2, md2, w2, label='Derrota', color=RED,  edgecolor=WHITE, linewidth=1.5)
ax.set_xticks(xp2); ax.set_xticklabels(ol, rotation=15, ha='right')
ax.set_ylabel('Media'); ax.set_title('Objetivos por Resultado', fontweight='bold'); ax.legend()

ax = axes[1, 0]
fc = ['blueWins','blueGoldDiff','blueExperienceDiff','blueKills','blueDeaths',
      'blueTotalGold','blueDragons','blueFirstBlood','blueCSPerMin','kills_diff']
fc = [c for c in fc if c in df.columns]
lc = ['Wins','GoldDiff','ExpDiff','Kills','Deaths','TotalGold','Dragons','FBlood','CS/Min','KDiff']
cm3 = df[fc].corr()
mask = np.triu(np.ones_like(cm3, dtype=bool))
sns.heatmap(cm3, ax=ax, mask=mask, cmap='coolwarm', annot=True, fmt='.2f',
            linewidths=0.4, annot_kws={'size':7.5},
            xticklabels=lc[:len(fc)], yticklabels=lc[:len(fc)], center=0)
ax.set_title('Mapa de Correlacao', fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=8); ax.tick_params(axis='y', rotation=0, labelsize=8)

ax = axes[1, 1]
ct = df[fc].corr()['blueWins'].drop('blueWins').sort_values()
cl_v = [RED if v < 0 else BLUE for v in ct.values]
yp3  = range(len(ct))
ax.hlines(yp3, 0, ct.values, colors=cl_v, linewidth=2, alpha=0.7)
ax.scatter(ct.values, yp3, color=cl_v, s=70, zorder=5)
ax.axvline(0, color='#374151', linewidth=1.2, alpha=0.5)
ax.set_yticks(list(yp3)); ax.set_yticklabels(lc[1:len(fc)], fontsize=9)
ax.set_xlabel('Correlacao de Pearson'); ax.set_title('Correlacao com o Target', fontweight='bold')
ax.set_xlim(-0.85, 0.85)

plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'grafico_03_early_game.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_03 salvo')

# GRAFICO 04
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('O Perfil do Time Vencedor', fontsize=16, fontweight='bold', y=1.02)

ax = axes[0]
mts_d = {'Kills':'blueKills','Ouro':'blueTotalGold','GoldDiff':'blueGoldDiff',
          'Wards':'blueWardsPlaced','CS/Min':'blueCSPerMin','Dragoes':'blueDragons'}
mv3 = {k: df[df['blueWins']==1][v].mean() for k,v in mts_d.items()}
md3 = {k: df[df['blueWins']==0][v].mean() for k,v in mts_d.items()}
dp3 = {k: (mv3[k]-md3[k])/abs(md3[k])*100 for k in mts_d}
ks3 = list(dp3.keys()); vs3 = list(dp3.values())
cg3 = [GREEN if v > 0 else RED for v in vs3]
ax.barh(ks3, vs3, color=cg3, edgecolor=WHITE, linewidth=1.5, height=0.55)
ax.axvline(0, color='#374151', linewidth=1.5)
ax.set_xlabel('Diferenca % (Vitoria vs Derrota)'); ax.set_title('Vantagem Relativa', fontweight='bold')

ax = axes[1]
for res, col, lbl in [(1, BLUE,'Vitoria'), (0, RED,'Derrota')]:
    data = df[df['blueWins']==res]['blueTotalExperience'].values
    kde  = gaussian_kde(data, bw_method=0.15)
    xr   = np.linspace(data.min(), data.max(), 300)
    yk   = kde(xr)
    ax.plot(xr, yk, color=col, linewidth=2.5, label=lbl)
    ax.fill_between(xr, yk, alpha=0.18, color=col)
    ax.axvline(np.median(data), color=col, linestyle='--', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Experiencia Total'); ax.set_ylabel('Densidade')
ax.set_title('Distribuicao de Experiencia', fontweight='bold'); ax.legend()

ax = axes[2]
df['combo_obj'] = (df['blueFirstBlood'].astype(str)+'_FB__'+
                   df['blueDragons'].clip(0,1).astype(str)+'_DG__'+
                   df['blueHeralds'].clip(0,1).astype(str)+'_HR')
cwr = df.groupby('combo_obj')['blueWins'].agg(['mean','count']).reset_index()
cwr = cwr[cwr['count'] >= 150].sort_values('mean', ascending=True)
cwr['label'] = (cwr['combo_obj']
    .str.replace('1_FB','FB+').str.replace('0_FB','FB-')
    .str.replace('1_DG','DG+').str.replace('0_DG','DG-')
    .str.replace('1_HR','HR+').str.replace('0_HR','HR-').str.replace('__',' '))
cc4 = [GREEN if v >= 0.5 else RED for v in cwr['mean']]
ax.barh(cwr['label'], cwr['mean']*100, color=cc4, edgecolor=WHITE, linewidth=1.2, height=0.6)
ax.axvline(50, color='#374151', linewidth=1.5, linestyle='--', alpha=0.7)
ax.set_xlabel('Taxa de Vitoria (%)'); ax.set_title('Win Rate por Combinacao Objetivos', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'grafico_04_perfil_vencedor.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_04 salvo')
print('FASE 2 concluida!')

Gerando graficos da Fase 2...
  grafico_01 salvo
  grafico_02 salvo
  grafico_03 salvo
  grafico_04 salvo
FASE 2 concluida!


In [6]:
# CELULA 6 - FASE 3: Preprocessamento, Pipeline e Checkpoint BI
print('FASE 3 - Preprocessamento e Pipeline')

DROP_COLS = ['gameId','redDeaths','redKills','blueGoldPerMin','redGoldPerMin',
             'redGoldDiff','redExperienceDiff','blueTotalMinionsKilled',
             'redTotalMinionsKilled','blue_total_cs','red_total_cs',
             'categoria_gold_diff','dragon_cat','combo_obj']
DROP_COLS = [c for c in DROP_COLS if c in df.columns]
df_clean  = df.drop(columns=DROP_COLS).copy()

for col in ['blueWardsPlaced','redWardsPlaced','blueWardsDestroyed','redWardsDestroyed']:
    if col in df_clean.columns:
        q05, q95 = df_clean[col].quantile(0.05), df_clean[col].quantile(0.95)
        df_clean[col] = df_clean[col].clip(q05, q95)

TARGET   = 'blueWins'
FEATURES = [c for c in df_clean.columns if c != TARGET]
X        = df_clean[FEATURES].select_dtypes(include=np.number).fillna(0)
y        = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

pipe_sc  = Pipeline([('scaler', StandardScaler())])
X_tr_sc  = pipe_sc.fit_transform(X_train)
X_te_sc  = pipe_sc.transform(X_test)

pca_full = PCA(random_state=42).fit(X_tr_sc)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_)
n_95     = int(np.argmax(cumvar >= 0.95) + 1)

pipe_final = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=n_95, random_state=42))])
X_tr_pca   = pipe_final.fit_transform(X_train)
X_te_pca   = pipe_final.transform(X_test)

np.save(os.path.join(DIRS['data'], 'X_train_scaled.npy'), X_tr_sc)
np.save(os.path.join(DIRS['data'], 'X_test_scaled.npy'),  X_te_sc)
np.save(os.path.join(DIRS['data'], 'X_train_pca.npy'),    X_tr_pca)
np.save(os.path.join(DIRS['data'], 'X_test_pca.npy'),     X_te_pca)
X_train.to_parquet(os.path.join(DIRS['data'], 'X_train.parquet'))
X_test.to_parquet( os.path.join(DIRS['data'], 'X_test.parquet'))
y_train.to_csv(os.path.join(DIRS['data'], 'y_train.csv'), index=False)
y_test.to_csv( os.path.join(DIRS['data'], 'y_test.csv'),  index=False)
joblib.dump(pipe_sc,    os.path.join(DIRS['models'], 'pipeline_scaler_only.pkl'))
joblib.dump(pipe_final, os.path.join(DIRS['models'], 'pipeline_scaler_pca.pkl'))

df_bi = df_clean.copy()
df_bi['resultado_label']   = df_bi['blueWins'].map({1:'Vitoria Azul', 0:'Derrota Azul'})
df_bi['categoria_gold']    = pd.cut(df_bi['blueGoldDiff'],
    bins=[-np.inf,-1500,0,1500,np.inf],
    labels=['Forte Desvantagem','Leve Desvantagem','Leve Vantagem','Forte Vantagem'])
df_bi['first_blood_label'] = df_bi['blueFirstBlood'].map({1:'Com First Blood', 0:'Sem First Blood'})
df_bi['dragon_label']      = pd.cut(df_bi['blueDragons'], bins=[-1,0,1,2,10],
    labels=['0 Dragoes','1 Dragao','2 Dragoes','3+'])
df_bi.to_csv(os.path.join(DIRS['data'], 'dataset_tratado_powerbi.csv'), index=False, encoding='utf-8-sig')
print('Checkpoint BI salvo:', df_bi.shape)

def calc_vif(Xdf):
    res = []
    Xa  = Xdf.values
    for i, col in enumerate(Xdf.columns):
        others = np.delete(Xa, i, axis=1)
        lr2    = LinearRegression().fit(others, Xa[:, i])
        r2     = lr2.score(others, Xa[:, i])
        res.append({'feature': col, 'VIF': round(1/(1-r2) if r2 < 1.0 else np.inf, 2)})
    return pd.DataFrame(res).sort_values('VIF', ascending=False)

vif_df = calc_vif(X.head(2000))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('Fase 3 - Multicolinearidade, VIF e Outliers', fontsize=15, fontweight='bold', y=1.02)

ax = axes[0]
fh = ['blueGoldDiff','blueExperienceDiff','blueKills','blueDeaths','blueTotalGold',
      'blueDragons','blueFirstBlood','blueWardsPlaced','blueCSPerMin','kills_diff']
fh = [c for c in fh if c in df_clean.columns]
lh = ['GoldDiff','ExpDiff','Kills','Deaths','TotalGold','Dragons','FBlood','Wards','CS/Min','KillsDiff']
mask2 = np.triu(np.ones((len(fh), len(fh)), dtype=bool))
sns.heatmap(df_clean[fh].corr(), ax=ax, mask=mask2, cmap='coolwarm', annot=True, fmt='.2f',
            linewidths=0.4, annot_kws={'size':7}, xticklabels=lh[:len(fh)], yticklabels=lh[:len(fh)], center=0)
ax.set_title('Correlacao pos-selecao', fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=7.5); ax.tick_params(axis='y', rotation=0, labelsize=7.5)

ax = axes[1]
vp = vif_df[vif_df['VIF'] < 100].head(20).sort_values('VIF')
cv2 = [RED if v > 10 else GOLD if v > 5 else GREEN for v in vp['VIF']]
ax.barh(vp['feature'], vp['VIF'], color=cv2, edgecolor=WHITE, linewidth=1, height=0.65)
ax.axvline(10, color=RED, linestyle='--', linewidth=1.5, label='VIF>10 (critico)')
ax.axvline(5,  color=GOLD, linestyle='--', linewidth=1.5, label='VIF>5 (atencao)')
ax.set_xlabel('VIF'); ax.set_title('VIF por Feature', fontweight='bold'); ax.legend(fontsize=8.5)

ax = axes[2]
before5 = df['blueWardsPlaced']; after5 = df_clean['blueWardsPlaced']
bp4 = ax.boxplot([before5, after5], patch_artist=True, widths=0.4,
                  medianprops=dict(color=WHITE, linewidth=2.5),
                  flierprops=dict(marker='o', markersize=3, alpha=0.3))
bp4['boxes'][0].set_facecolor(RED); bp4['boxes'][0].set_alpha(0.7)
bp4['boxes'][1].set_facecolor(GREEN); bp4['boxes'][1].set_alpha(0.7)
for w in bp4['whiskers'] + bp4['caps']: w.set_color(GRAY)
ax.set_xticks([1, 2]); ax.set_xticklabels(['Antes', 'Depois (Winsorized)'])
ax.set_ylabel('blueWardsPlaced'); ax.set_title('Tratamento de Outliers', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'grafico_05_multicolinearidade_vif.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_05 salvo')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.suptitle('Fase 3 - PCA: Variancia Explicada', fontsize=14, fontweight='bold', y=1.02)

ax = axes[0]
vi = pca_full.explained_variance_ratio_[:25] * 100
ax.bar(range(1, 26), vi, color=BLUE, alpha=0.75, edgecolor=WHITE, linewidth=0.8)
ax.set_xlabel('Componente Principal'); ax.set_ylabel('Variancia (%)')
ax.set_title('Variancia por Componente', fontweight='bold')

ax = axes[1]
ax.plot(range(1, len(cumvar)+1), cumvar*100, color=BLUE, linewidth=2.5, marker='o', markersize=3, markevery=2)
ax.axhline(95, color=RED, linestyle='--', linewidth=1.5, label='95%')
ax.axvline(n_95, color=RED, linestyle=':', linewidth=1.5)
ax.fill_between(range(1, len(cumvar)+1), cumvar*100, alpha=0.08, color=BLUE)
ax.set_xlabel('Num. Componentes'); ax.set_ylabel('Variancia Acumulada (%)')
ax.set_title('Variancia Acumulada', fontweight='bold'); ax.legend(); ax.set_ylim(40, 103)
ax.text(n_95+0.3, 88, str(n_95)+' comps -> 95%', color=RED, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'grafico_06_pca_variancia.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_06 salvo')
print('FASE 3 concluida! PCA:', n_95, 'componentes')

FASE 3 - Preprocessamento e Pipeline
Checkpoint BI salvo: (9879, 39)
  grafico_05 salvo
  grafico_06 salvo
FASE 3 concluida! PCA: 17 componentes


In [7]:
# CELULA 7 - FASE 4: K-Means
print('FASE 4 - K-Means Clustering')

K_RANGE     = range(2, 11)
inertias    = []
silhouettes = []
for k in K_RANGE:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    lbl = km.fit_predict(X_tr_pca)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_tr_pca, lbl, sample_size=2000, random_state=42)
    silhouettes.append(sil)
    print('  K=' + str(k) + ' | Inertia=' + str(round(km.inertia_, 0)) + ' | Sil=' + str(round(sil, 4)))

K_FINAL      = 4
km_final     = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20, max_iter=500)
labels_train = km_final.fit_predict(X_tr_pca)
sil_final    = silhouette_score(X_tr_pca, labels_train, sample_size=3000, random_state=42)
joblib.dump(km_final, os.path.join(DIRS['models'], 'kmeans_k4.pkl'))
np.save(os.path.join(DIRS['data'], 'cluster_labels_train.npy'), labels_train)

df_cl = X_train.copy()
df_cl['cluster']  = labels_train
df_cl['blueWins'] = y_train.values
PCOLS   = ['blueGoldDiff','blueKills','blueDeaths','blueTotalGold','blueDragons',
           'blueFirstBlood','blueCSPerMin','blueAvgLevel','kills_diff']
PCOLS   = [c for c in PCOLS if c in df_cl.columns]
profile = df_cl.groupby('cluster')[PCOLS + ['blueWins']].mean().round(3)
profile['n_partidas']       = df_cl.groupby('cluster').size()
profile['taxa_vitoria_pct'] = (profile['blueWins'] * 100).round(1)
profile = profile.drop(columns='blueWins')
profile.to_csv(os.path.join(DIRS['data'], 'cluster_profile.csv'))

CNAMES = {}
for cl in range(K_FINAL):
    wr = profile.loc[cl, 'taxa_vitoria_pct']
    if wr >= 65:   CNAMES[cl] = 'C' + str(cl) + ' Dominante Azul'
    elif wr >= 52: CNAMES[cl] = 'C' + str(cl) + ' Vantagem Leve'
    elif wr >= 45: CNAMES[cl] = 'C' + str(cl) + ' Equilibrada'
    else:          CNAMES[cl] = 'C' + str(cl) + ' Desvantagem Azul'

print('Clusters identificados:')
for i in range(K_FINAL):
    print(' ', CNAMES[i], '| Win Rate:', profile.loc[i,'taxa_vitoria_pct'], '%')

ks = list(K_RANGE)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
fig.suptitle('Fase 4 - Elbow e Silhouette', fontsize=14, fontweight='bold', y=1.02)

ax = axes[0]
ax.plot(ks, inertias, color=BLUE, linewidth=2.5, marker='o', markersize=8,
        markerfacecolor=WHITE, markeredgewidth=2.5, markeredgecolor=BLUE)
ax.fill_between(ks, inertias, alpha=0.07, color=BLUE)
ax.axvline(K_FINAL, color=RED, linestyle='--', linewidth=2, label='K=' + str(K_FINAL))
ax.set_xlabel('K'); ax.set_ylabel('Inertia'); ax.set_title('Elbow Method', fontweight='bold')
ax.set_xticks(ks); ax.legend()

ax = axes[1]
cs2  = [RED if k == K_FINAL else BLUE for k in ks]
bars4 = ax.bar(ks, silhouettes, color=cs2, edgecolor=WHITE, linewidth=1.5, width=0.6)
ax.set_xlabel('K'); ax.set_ylabel('Silhouette Score'); ax.set_title('Silhouette por K', fontweight='bold')
ax.set_xticks(ks); ax.set_ylim(0, max(silhouettes) * 1.25)
ax.axvline(K_FINAL, color=RED, linestyle='--', linewidth=2)
for b, v in zip(bars4, silhouettes):
    ax.text(b.get_x()+b.get_width()/2, v+0.002, str(round(v,3)), ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'grafico_07_elbow_silhouette.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_07 salvo')

fig = plt.figure(figsize=(18, 12), facecolor=BG)
fig.suptitle('Fase 4 - Perfil dos Clusters', fontsize=16, fontweight='bold', y=0.99)
gs4 = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.38)

ax = fig.add_subplot(gs4[0, 0])
wr_v2 = [profile.loc[i, 'taxa_vitoria_pct'] for i in range(K_FINAL)]
brs2  = ax.bar(range(K_FINAL), wr_v2, color=CLUSTER_COLORS, edgecolor=WHITE, linewidth=2, width=0.6)
ax.axhline(50, color=GRAY, linestyle='--', linewidth=1.5); ax.set_ylim(0, 95)
ax.set_xticks(range(K_FINAL)); ax.set_xticklabels(['C'+str(i) for i in range(K_FINAL)])
ax.set_ylabel('Win Rate (%)'); ax.set_title('Win Rate por Cluster', fontweight='bold')
for b, v in zip(brs2, wr_v2):
    ax.text(b.get_x()+b.get_width()/2, v+1.5, str(v)+'%', ha='center', fontweight='bold', fontsize=12)

ax = fig.add_subplot(gs4[0, 1])
nv2 = [profile.loc[i, 'n_partidas'] for i in range(K_FINAL)]
wedges2, _, autos2 = ax.pie(nv2, colors=CLUSTER_COLORS, startangle=90, autopct='%1.1f%%',
    pctdistance=0.78, wedgeprops=dict(edgecolor=WHITE, linewidth=2.5))
for at in autos2: at.set_fontsize(10); at.set_fontweight('bold'); at.set_color(WHITE)
ax.legend(wedges2, ['C'+str(i)+': '+str(int(n)) for i,n in enumerate(nv2)],
          loc='lower center', bbox_to_anchor=(0.5,-0.2), fontsize=9, ncol=2)
ax.set_title('Distribuicao de Partidas', fontweight='bold')

ax = fig.add_subplot(gs4[0, 2])
gd2 = [profile.loc[i, 'blueGoldDiff'] for i in range(K_FINAL)]
cg2 = [GREEN if v >= 0 else RED for v in gd2]
ax.barh(range(K_FINAL), gd2, color=cg2, edgecolor=WHITE, linewidth=1.5, height=0.55)
ax.axvline(0, color='#374151', linewidth=1.5)
ax.set_yticks(range(K_FINAL)); ax.set_yticklabels(['C'+str(i) for i in range(K_FINAL)])
ax.set_xlabel('Gold Diff medio'); ax.set_title('Gold Diff por Cluster', fontweight='bold')

ax = fig.add_subplot(gs4[1, :2])
rcols = ['blueKills','blueDeaths','blueDragons','blueFirstBlood','blueCSPerMin','blueGoldDiff']
rcols = [c for c in rcols if c in profile.columns]
rlbls = ['Kills','Deaths','Dragons','FBlood','CS/Min','GoldDiff']
np3   = profile[rcols].copy()
for col in rcols:
    mn2, mx2 = np3[col].min(), np3[col].max()
    np3[col] = (np3[col]-mn2)/(mx2-mn2+1e-9)
xp4 = np.arange(len(rcols)); ww3 = 0.2
for i in range(K_FINAL):
    ax.bar(xp4+i*ww3-(K_FINAL-1)*ww3/2, np3.loc[i,rcols].values, ww3,
           label='C'+str(i), color=CLUSTER_COLORS[i], edgecolor=WHITE, alpha=0.85)
ax.set_xticks(xp4); ax.set_xticklabels(rlbls[:len(rcols)], fontsize=10)
ax.set_ylabel('Valor Normalizado (0-1)'); ax.set_title('Comparacao por Cluster', fontweight='bold')
ax.legend(fontsize=9, ncol=K_FINAL)

ax = fig.add_subplot(gs4[1, 2])
dbp2 = [df_cl[df_cl['cluster']==i]['blueGoldDiff'].values for i in range(K_FINAL)]
bp5  = ax.boxplot(dbp2, patch_artist=True, widths=0.5,
                   medianprops=dict(color=WHITE, linewidth=2.5),
                   flierprops=dict(marker='o', markersize=2, alpha=0.2))
for box, col in zip(bp5['boxes'], CLUSTER_COLORS): box.set_facecolor(col); box.set_alpha(0.8)
for w in bp5['whiskers'] + bp5['caps']: w.set_color(GRAY)
ax.set_xticks(range(1, K_FINAL+1)); ax.set_xticklabels(['C'+str(i) for i in range(K_FINAL)])
ax.set_ylabel('Gold Diff'); ax.set_title('Gold Diff por Cluster', fontweight='bold')
ax.axhline(0, color='#374151', linewidth=1.2, linestyle=':', alpha=0.7)

plt.savefig(os.path.join(FIGS, 'grafico_08_cluster_perfil.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_08 salvo')

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
fig.suptitle('Fase 4 - Silhouette e Projecao PCA', fontsize=14, fontweight='bold', y=1.02)

ax = axes[0]
ss2 = silhouette_samples(X_tr_pca, labels_train)
yl2 = 10
for i in range(K_FINAL):
    si2 = np.sort(ss2[labels_train==i])
    yu2 = yl2 + si2.shape[0]
    ax.fill_betweenx(np.arange(yl2, yu2), 0, si2, facecolor=CLUSTER_COLORS[i], alpha=0.75)
    ax.text(-0.05, yl2+0.5*si2.shape[0], 'C'+str(i), fontsize=10, fontweight='bold', color=CLUSTER_COLORS[i])
    yl2 = yu2 + 10
ax.axvline(sil_final, color=RED, linestyle='--', linewidth=2, label='Media='+str(round(sil_final,3)))
ax.set_xlabel('Silhouette Coefficient'); ax.set_title('Silhouette Plot', fontweight='bold')
ax.legend(fontsize=9); ax.set_yticks([])

ax = axes[1]
si3 = np.random.choice(len(X_tr_pca), size=2000, replace=False)
xs2 = X_tr_pca[si3]; ls2 = labels_train[si3]
for i in range(K_FINAL):
    mk2 = ls2 == i
    ax.scatter(xs2[mk2, 0], xs2[mk2, 1], color=CLUSTER_COLORS[i], alpha=0.4, s=15, label='C'+str(i))
c2d2 = km_final.cluster_centers_[:, :2]
for i, ct2 in enumerate(c2d2):
    ax.scatter(*ct2, color=CLUSTER_COLORS[i], s=200, marker='*', edgecolors='#1E293B', linewidths=1.5, zorder=10)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title('Projecao PCA', fontweight='bold')
ax.legend(fontsize=8.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'grafico_09_silhouette_pca_scatter.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_09 salvo')
print('FASE 4 concluida!')

FASE 4 - K-Means Clustering
  K=2 | Inertia=205688.0 | Sil=0.1734
  K=3 | Inertia=191360.0 | Sil=0.1039
  K=4 | Inertia=179404.0 | Sil=0.1068
  K=5 | Inertia=173180.0 | Sil=0.109
  K=6 | Inertia=167059.0 | Sil=0.1078
  K=7 | Inertia=162627.0 | Sil=0.0899
  K=8 | Inertia=157832.0 | Sil=0.0822
  K=9 | Inertia=152568.0 | Sil=0.0864
  K=10 | Inertia=148426.0 | Sil=0.0871
Clusters identificados:
  C0 Desvantagem Azul | Win Rate: 14.0 %
  C1 Dominante Azul | Win Rate: 85.4 %
  C2 Equilibrada | Win Rate: 46.1 %
  C3 Equilibrada | Win Rate: 51.5 %
  grafico_07 salvo
  grafico_08 salvo
  grafico_09 salvo
FASE 4 concluida!


In [8]:
# CELULA 8 - FASE 5: Modelagem Supervisionada
print('FASE 5 - Batalha dos Modelos')

cv5   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Xtr   = X_train.values; Xte = X_test.values
results = []

def evaluate(name, model, Xtr_, Xte_, ytr, yte):
    model.fit(Xtr_, ytr)
    yp   = model.predict(Xte_)
    ypr  = model.predict_proba(Xte_)[:,1] if hasattr(model,'predict_proba') else model.decision_function(Xte_)
    cvau = cross_val_score(model, Xtr_, ytr, cv=cv5, scoring='roc_auc').mean()
    return dict(model=name, acc=round(accuracy_score(yte,yp)*100,2),
                auc=round(roc_auc_score(yte,ypr)*100,2), f1=round(f1_score(yte,yp)*100,2),
                prec=round(precision_score(yte,yp)*100,2), rec=round(recall_score(yte,yp)*100,2),
                cv_auc=round(cvau*100,2), fitted=model)

print('Treinando...')
results.append(evaluate('Naive Bayes',        GaussianNB(), X_tr_sc, X_te_sc, y_train, y_test)); print('  NB ok')
results.append(evaluate('Regressao Logistica',LogisticRegression(max_iter=1000,random_state=42), X_tr_sc, X_te_sc, y_train, y_test)); print('  LR ok')
results.append(evaluate('Arvore de Decisao',  DecisionTreeClassifier(random_state=42), Xtr, Xte, y_train, y_test)); print('  DT ok')
results.append(evaluate('Random Forest',      RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1), Xtr, Xte, y_train, y_test)); print('  RF ok')
results.append(evaluate('XGBoost',            XGBClassifier(n_estimators=200,learning_rate=0.1,max_depth=5,random_state=42,eval_metric='logloss',verbosity=0), Xtr, Xte, y_train, y_test)); print('  XGB ok')
results.append(evaluate('SVM',                SVC(kernel='rbf',probability=True,random_state=42), X_tr_sc, X_te_sc, y_train, y_test)); print('  SVM ok')

print('GridSearchCV...')
gs5 = GridSearchCV(XGBClassifier(random_state=42,eval_metric='logloss',verbosity=0),
    {'n_estimators':[100,200,300],'max_depth':[3,5,7],'learning_rate':[0.05,0.1,0.2]},
    cv=cv5, scoring='roc_auc', n_jobs=-1)
gs5.fit(Xtr, y_train)
best_mdl  = gs5.best_estimator_
yp_best   = best_mdl.predict(Xte)
ypr_best  = best_mdl.predict_proba(Xte)[:,1]
tuned_res = dict(model='XGBoost (Tuned)',
    acc=round(accuracy_score(y_test,yp_best)*100,2),
    auc=round(roc_auc_score(y_test,ypr_best)*100,2),
    f1=round(f1_score(y_test,yp_best)*100,2),
    prec=round(precision_score(y_test,yp_best)*100,2),
    rec=round(recall_score(y_test,yp_best)*100,2),
    cv_auc=round(gs5.best_score_*100,2), fitted=best_mdl)
print('Best params:', gs5.best_params_)

all_res = [{k:v for k,v in r.items() if k!='fitted'} for r in results]
all_res.append({k:v for k,v in tuned_res.items() if k!='fitted'})
df_res = pd.DataFrame(all_res)
df_res.to_csv(os.path.join(DIRS['data'], 'model_results.csv'), index=False)

for r in results:
    fn = r['model'].lower().replace(' ','_')
    joblib.dump(r['fitted'], os.path.join(DIRS['models'], fn+'.pkl'))
joblib.dump(best_mdl, os.path.join(DIRS['models'], 'xgboost_tuned.pkl'))

lr_model = results[1]['fitted']
np.save(os.path.join(DIRS['data'], 'y_pred_best.npy'),  lr_model.predict(X_te_sc))
np.save(os.path.join(DIRS['data'], 'y_proba_best.npy'), lr_model.predict_proba(X_te_sc)[:,1])
np.save(os.path.join(DIRS['data'], 'y_test_array.npy'), y_test.values)

print(df_res[['model','cv_auc','auc','acc','f1']].sort_values('cv_auc',ascending=False).to_string(index=False))

NAMES_S = ['NB','LogReg','Tree','RF','XGB','SVM','XGB*']
MC      = [TEAL, BLUE, ORANGE, GREEN, GOLD, PURPLE, RED]
dp2     = pd.DataFrame(all_res)

fig = plt.figure(figsize=(18, 12), facecolor=BG)
fig.suptitle('Fase 5 - Batalha dos Modelos', fontsize=16, fontweight='bold', y=0.99)
gs6 = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)
mets6 = [('cv_auc','CV AUC (%)'),('auc','AUC ROC (%)'),('acc','Acuracia (%)'),
          ('f1','F1 (%)'),('prec','Precisao (%)'),('rec','Recall (%)')]
for idx, (col, lbl) in enumerate(mets6):
    ax = fig.add_subplot(gs6[idx//3, idx%3])
    vs6 = dp2[col].values
    bv  = max(vs6)
    cc6 = [RED if 'Tuned' in n else BLUE if v==bv else GRAY for n,v in zip(dp2['model'],vs6)]
    bars6 = ax.bar(NAMES_S[:len(vs6)], vs6, color=cc6, edgecolor=WHITE, linewidth=1.5, width=0.65)
    ax.set_ylabel(lbl); ax.set_title(lbl, fontweight='bold'); ax.set_ylim(min(vs6)*0.92, max(vs6)*1.08)
    for b, v in zip(bars6, vs6):
        ax.text(b.get_x()+b.get_width()/2, v+0.2, str(v), ha='center', fontsize=8.5, fontweight='bold')
plt.savefig(os.path.join(FIGS,'grafico_10_batalha_modelos.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_10 salvo')

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('Fase 5 - Duelos Diretos', fontsize=15, fontweight='bold', y=1.02)
dd2  = dp2.set_index('model')
mts7 = ['cv_auc','auc','acc','f1']; xl = ['CV AUC','AUC','Acc','F1']
duelos7 = [
    ('Duelo 1: Arvore vs LogReg', ['Arvore de Decisao','Regressao Logistica'],[ORANGE,BLUE]),
    ('Duelo 2: Arvore vs RF',     ['Arvore de Decisao','Random Forest'],[ORANGE,GREEN]),
    ('Duelo 3: RF vs XGB vs XGB*',['Random Forest','XGBoost','XGBoost (Tuned)'],[GREEN,GOLD,RED]),
]
for ax, (title, models, colors) in zip(axes, duelos7):
    xp5 = np.arange(len(mts7)); ww4 = 0.25 if len(models)==3 else 0.35
    for i, (mdl, col) in enumerate(zip(models, colors)):
        if mdl not in dd2.index: continue
        vs7    = [dd2.loc[mdl, m] for m in mts7]
        offset = (i-(len(models)-1)/2)*ww4
        bars7  = ax.bar(xp5+offset, vs7, ww4, label=mdl, color=col, edgecolor=WHITE, alpha=0.88)
        for b, v in zip(bars7, vs7):
            ax.text(b.get_x()+b.get_width()/2, v+0.3, str(v), ha='center', fontsize=7.5, fontweight='bold')
    ax.set_xticks(xp5); ax.set_xticklabels(xl, fontsize=9)
    ax.set_ylabel('Score (%)'); ax.set_ylim(55, 90)
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGS,'grafico_11_duelos_roc.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_11 salvo')

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('Fase 5 - ROC, Ranking e Tabela', fontsize=14, fontweight='bold', y=1.02)
ax = axes[0]
ri = [('LogReg','regressao_logistica',BLUE,X_te_sc),('RF','random_forest',GREEN,Xte),
      ('XGB','xgboost',GOLD,Xte),('XGB*','xgboost_tuned',RED,Xte),('SVM','svm',PURPLE,X_te_sc)]
for nm, fn, col, Xte_ in ri:
    mp = os.path.join(DIRS['models'], fn+'.pkl')
    if not os.path.exists(mp): continue
    ml  = joblib.load(mp)
    yp8 = ml.predict_proba(Xte_)[:,1]
    fp, tp, _ = roc_curve(y_test.values, yp8)
    av = roc_auc_score(y_test.values, yp8)
    ls = '-' if nm in ['LogReg','XGB*'] else '--'
    lw = 2.5 if nm in ['LogReg','XGB*'] else 1.5
    ax.plot(fp, tp, color=col, linewidth=lw, linestyle=ls, label=nm+' ('+str(round(av*100,1))+'%)')
ax.plot([0,1],[0,1], color=GRAY, linewidth=1.2, linestyle=':', alpha=0.7)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('Curva ROC', fontweight='bold'); ax.legend(fontsize=8)
ax = axes[1]
dr2 = dp2.sort_values('cv_auc', ascending=True)
nr2 = [n.replace('Regressao Logistica','LogReg').replace('Arvore de Decisao','Tree')
        .replace('Random Forest','RF').replace('XGBoost (Tuned)','XGB*') for n in dr2['model']]
cr2 = [RED if 'Tuned' in n else BLUE if 'Logistica' in n else GRAY for n in dr2['model']]
ax.barh(nr2, dr2['cv_auc'], color=cr2, edgecolor=WHITE, linewidth=1.2, height=0.65)
ax.set_xlabel('CV AUC (%)'); ax.set_title('Ranking Final', fontweight='bold'); ax.set_xlim(55, 88)
for i, val in enumerate(dr2['cv_auc']):
    ax.text(val+0.2, i, str(val)+'%', va='center', fontsize=9, fontweight='bold')
ax = axes[2]; ax.axis('off')
dt3 = dp2.sort_values('cv_auc', ascending=False).copy()
dt3['model'] = dt3['model'].str.replace('Regressao Logistica','LogReg').str.replace('Arvore de Decisao','Tree').str.replace('Random Forest','RF')
cc9 = ['model','cv_auc','auc','acc','f1']; ll9 = ['Modelo','CV AUC','AUC','Acc','F1']
tbl3 = [[str(v) for v in row] for row in dt3[cc9].values]
table3 = ax.table(cellText=tbl3, colLabels=ll9, cellLoc='center', loc='center', bbox=[0,0,1,1])
table3.auto_set_font_size(False); table3.set_fontsize(9)
for (r,c), cell in table3.get_celld().items():
    cell.set_edgecolor('#E5E7EB')
    if r==0: cell.set_facecolor('#1E293B'); cell.set_text_props(color=WHITE, fontweight='bold')
    elif r==1: cell.set_facecolor('#DCFCE7'); cell.set_text_props(fontweight='bold')
    else: cell.set_facecolor('#F8FAFC' if r%2==0 else WHITE)
ax.set_title('Tabela Final', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGS,'grafico_12_roc_ranking_tabela.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_12 salvo')
print('FASE 5 concluida!')

FASE 5 - Batalha dos Modelos
Treinando...
  NB ok
  LR ok
  DT ok
  RF ok
  XGB ok
  SVM ok
GridSearchCV...
Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
              model  cv_auc   auc   acc    f1
Regressao Logistica   80.87 81.06 73.38 73.03
    XGBoost (Tuned)   80.65 81.07 73.43 72.95
        Naive Bayes   79.93 79.51 72.98 72.56
      Random Forest   79.85 79.98 72.17 71.68
            XGBoost   79.27 79.75 71.51 70.96
                SVM   78.54 78.79 72.67 72.16
  Arvore de Decisao   62.84 64.02 64.02 63.71
  grafico_10 salvo
  grafico_11 salvo
  grafico_12 salvo
FASE 5 concluida!


In [9]:
# CELULA 9 - FASE 6: Avaliacao e Interpretabilidade
print('FASE 6 - Avaliacao Pos-Modelo')

y_pred9  = lr_model.predict(X_te_sc)
y_proba9 = lr_model.predict_proba(X_te_sc)[:,1]
cm9      = confusion_matrix(y_test, y_pred9)
print(classification_report(y_test, y_pred9, target_names=['Derrota','Vitoria']))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor=BG)
fig.suptitle('Fase 6 - Avaliacao: Regressao Logistica', fontsize=15, fontweight='bold', y=1.02)

ax = axes[0]
cmn9 = cm9.astype('float') / cm9.sum(axis=1)[:, np.newaxis]
NL   = chr(10)
lblcm9 = [
    ['TN' + NL + 'Derrota' + NL + 'correta', 'FP' + NL + 'Falso' + NL + 'Alerta'],
    ['FN' + NL + 'Vitoria' + NL + 'Perdida', 'TP' + NL + 'Vitoria' + NL + 'correta']
]
colcm9 = [[BLUE, RED], [RED, GREEN]]
for i in range(2):
    for j in range(2):
        ax.add_patch(plt.Rectangle((j-0.5, 1-i-0.5), 1, 1,
                     facecolor=colcm9[i][j], alpha=0.75, zorder=1))
        ax.text(j, 1-i,
                str(cm9[i,j]) + chr(10) + '(' + str(round(cmn9[i,j]*100,1)) + '%)',
                ha='center', va='center', fontsize=13, fontweight='bold', color=WHITE, zorder=2)
        ax.text(j, 1-i+0.32, lblcm9[i][j],
                ha='center', va='center', fontsize=8, color=WHITE, alpha=0.9, zorder=2)
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Previsto: Derrota', 'Previsto: Vitoria'], fontsize=10)
ax.set_yticklabels(['Real: Vitoria', 'Real: Derrota'], fontsize=10)
ax.set_title('Matriz de Confusao', fontweight='bold'); ax.grid(False)

ax = axes[1]
rep9  = classification_report(y_test, y_pred9, target_names=['Derrota','Vitoria'], output_dict=True)
clss9 = ['Derrota', 'Vitoria']
mtr9  = ['precision', 'recall', 'f1-score']
ltr9  = ['Precisao', 'Recall', 'F1']
xp6   = np.arange(len(mtr9)); w6 = 0.3
for i, (cls, col) in enumerate(zip(clss9, [RED, BLUE])):
    vs9   = [rep9[cls][m] * 100 for m in mtr9]
    bars9 = ax.bar(xp6 + (i-0.5)*w6, vs9, w6, label=cls,
                   color=col, edgecolor=WHITE, linewidth=1.5, alpha=0.88)
    for b, v in zip(bars9, vs9):
        ax.text(b.get_x()+b.get_width()/2, v+0.5,
                str(round(v,1))+'%', ha='center', fontsize=9, fontweight='bold')
acc_v9 = rep9['accuracy'] * 100
ax.axhline(acc_v9, color=GOLD, linestyle='--', linewidth=1.8,
           label='Acuracia: ' + str(round(acc_v9,1)) + '%')
ax.set_xticks(xp6); ax.set_xticklabels(ltr9, fontsize=10)
ax.set_ylabel('Score (%)'); ax.set_ylim(60, 85)
ax.set_title('Precision / Recall / F1', fontweight='bold'); ax.legend(fontsize=8.5)

ax = axes[2]
fpr9, tpr9, thr9 = roc_curve(y_test, y_proba9)
auc_v9 = roc_auc_score(y_test, y_proba9)
ax.plot(fpr9, tpr9, color=BLUE, linewidth=2.5,
        label='LogReg (AUC=' + str(round(auc_v9*100,2)) + '%)')
ax.fill_between(fpr9, tpr9, alpha=0.08, color=BLUE)
ax.plot([0,1], [0,1], color=GRAY, linewidth=1.5, linestyle='--', alpha=0.6, label='Aleatorio')
oi9 = np.argmin(np.sqrt((1-tpr9)**2 + fpr9**2))
ax.scatter(fpr9[oi9], tpr9[oi9], color=RED, s=120, zorder=5,
           label='Limiar=' + str(round(thr9[oi9],2)))
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Curva ROC', fontweight='bold'); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIGS,'grafico_13_confusao_report_roc.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_13 salvo')

fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor=BG)
fig.suptitle('Fase 6 - Feature Importance', fontsize=14, fontweight='bold', y=1.02)

fn3  = X_test.columns.tolist()
coef = lr_model.coef_[0]
fi   = pd.DataFrame({'feature': fn3, 'coef': coef})
fi['abs_coef'] = fi['coef'].abs()
fi   = fi.sort_values('abs_coef', ascending=True).tail(20)

ax = axes[0]
cfi = [GREEN if v > 0 else RED for v in fi['coef']]
ax.barh(fi['feature'], fi['coef'], color=cfi, edgecolor=WHITE, linewidth=1, height=0.7, alpha=0.85)
ax.axvline(0, color='#374151', linewidth=1.5)
ax.set_xlabel('Coeficiente (Log-Odds)'); ax.set_title('Coeficientes por Feature', fontweight='bold')

ax = axes[1]
fa3  = fi.sort_values('abs_coef', ascending=True)
cfa3 = [GREEN if v > 0 else RED for v in fa3['coef']]
ax.barh(fa3['feature'], fa3['abs_coef'], color=cfa3, edgecolor=WHITE, linewidth=1, height=0.7, alpha=0.85)
ax.set_xlabel('|Coeficiente|'); ax.set_title('Importancia Absoluta (Top 20)', fontweight='bold')
le3 = [mpatches.Patch(facecolor=GREEN, label='Favorece Vitoria'),
       mpatches.Patch(facecolor=RED,   label='Favorece Derrota')]
ax.legend(handles=le3, fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIGS,'grafico_14_feature_importance.png'), dpi=150, bbox_inches='tight', facecolor=BG)
plt.close(); print('  grafico_14 salvo')
print('FASE 6 concluida!')

FASE 6 - Avaliacao Pos-Modelo
              precision    recall  f1-score   support

     Derrota       0.73      0.75      0.74       990
     Vitoria       0.74      0.72      0.73       986

    accuracy                           0.73      1976
   macro avg       0.73      0.73      0.73      1976
weighted avg       0.73      0.73      0.73      1976

  grafico_13 salvo
  grafico_14 salvo
FASE 6 concluida!


In [10]:
# CELULA 10 - FASE 7: Plano de BI e Formulas DAX
print('FASE 7 - Camada de Negocios e Business Intelligence')
print('=' * 55)

# KPIs calculados diretamente do dataset
df_bi2 = pd.read_csv(os.path.join(DIRS['data'], 'dataset_tratado_powerbi.csv'))

total       = len(df_bi2)
win_rate    = df_bi2['blueWins'].mean() * 100
gold_v      = df_bi2[df_bi2['blueWins']==1]['blueGoldDiff'].mean()
gold_d      = df_bi2[df_bi2['blueWins']==0]['blueGoldDiff'].mean()
fb_wr       = df_bi2[df_bi2['blueFirstBlood']==1]['blueWins'].mean() * 100
nfb_wr      = df_bi2[df_bi2['blueFirstBlood']==0]['blueWins'].mean() * 100
drag_wr     = df_bi2[df_bi2['blueDragons']>=1]['blueWins'].mean() * 100
nodrag_wr   = df_bi2[df_bi2['blueDragons']==0]['blueWins'].mean() * 100

print()
print('KPI CARDS PARA O POWER BI')
print('-' * 40)
print('Total de Partidas          :', total)
print('Win Rate Azul Global       :', round(win_rate,1), '%')
print('Gold Diff Medio (Vitorias) :', '+' + str(round(gold_v,0)))
print('Gold Diff Medio (Derrotas) :', str(round(gold_d,0)))
print('Win Rate COM First Blood   :', round(fb_wr,1), '%')
print('Win Rate SEM First Blood   :', round(nfb_wr,1), '%')
print('Impacto do First Blood     :', '+' + str(round(fb_wr-nfb_wr,1)), 'p.p.')
print('Win Rate COM Dragao        :', round(drag_wr,1), '%')
print('Win Rate SEM Dragao        :', round(nodrag_wr,1), '%')
print('Impacto do Dragao          :', '+' + str(round(drag_wr-nodrag_wr,1)), 'p.p.')

print()
print('WIN RATE POR CATEGORIA DE GOLD:')
wr_gold = df_bi2.groupby('categoria_gold')['blueWins'].mean() * 100
for cat, val in wr_gold.sort_values(ascending=False).items():
    print('  ' + str(cat) + ': ' + str(round(val,1)) + '%')

print()
print('WIN RATE POR DRAGON LABEL:')
wr_drag = df_bi2.groupby('dragon_label')['blueWins'].mean() * 100
for cat, val in wr_drag.items():
    print('  ' + str(cat) + ': ' + str(round(val,1)) + '%')

print()
print('=' * 55)
print('FORMULAS DAX PARA O POWER BI')
print('=' * 55)

dax = {
    '1. Taxa de Vitoria Azul (medida base)': '''
Taxa de Vitoria Azul =
DIVIDE(
    CALCULATE(COUNTROWS(lol_matches), lol_matches[blueWins] = 1),
    COUNTROWS(lol_matches),
    0
)''',

    '2. Taxa de Vitoria com First Blood': '''
Vitoria com First Blood =
CALCULATE(
    DIVIDE(
        CALCULATE(COUNTROWS(lol_matches), lol_matches[blueWins] = 1),
        COUNTROWS(lol_matches), 0
    ),
    lol_matches[blueFirstBlood] = 1
)''',

    '3. Impacto do First Blood (diferenca em p.p.)': '''
Impacto First Blood pp =
VAR ComFB =
    CALCULATE(
        DIVIDE(
            CALCULATE(COUNTROWS(lol_matches), lol_matches[blueWins] = 1),
            COUNTROWS(lol_matches), 0
        ),
        lol_matches[blueFirstBlood] = 1
    )
VAR SemFB =
    CALCULATE(
        DIVIDE(
            CALCULATE(COUNTROWS(lol_matches), lol_matches[blueWins] = 1),
            COUNTROWS(lol_matches), 0
        ),
        lol_matches[blueFirstBlood] = 0
    )
RETURN (ComFB - SemFB) * 100''',

    '4. Gold Diff Medio por Resultado': '''
Gold Diff Medio =
AVERAGEX(lol_matches, lol_matches[blueGoldDiff])''',

    '5. Win Rate por Faixa de Gold (tabela calculada)': '''
WinRate por Gold Faixa =
SUMMARIZE(
    lol_matches,
    lol_matches[categoria_gold],
    "Total", COUNTROWS(lol_matches),
    "Vitorias", CALCULATE(COUNTROWS(lol_matches), lol_matches[blueWins] = 1),
    "Win Rate %",
        DIVIDE(
            CALCULATE(COUNTROWS(lol_matches), lol_matches[blueWins] = 1),
            COUNTROWS(lol_matches), 0
        ) * 100
)''',

    '6. Win Rate por Numero de Dragoes': '''
WinRate por Dragao =
SUMMARIZE(
    lol_matches,
    lol_matches[dragon_label],
    "Total", COUNTROWS(lol_matches),
    "Win Rate %",
        DIVIDE(
            CALCULATE(COUNTROWS(lol_matches), lol_matches[blueWins] = 1),
            COUNTROWS(lol_matches), 0
        ) * 100
)''',

    '7. Comparacao de KDA Vencedores vs Perdedores': '''
KDA Ratio Vencedores =
CALCULATE(AVERAGE(lol_matches[blue_kda_ratio]), lol_matches[blueWins] = 1)

KDA Ratio Perdedores =
CALCULATE(AVERAGE(lol_matches[blue_kda_ratio]), lol_matches[blueWins] = 0)

Diferenca KDA =
[KDA Ratio Vencedores] - [KDA Ratio Perdedores]''',
}

for titulo, formula in dax.items():
    print()
    print('DAX', titulo)
    print(formula)

print()
print('=' * 55)
print('DASHBOARDS RECOMENDADOS PARA O POWER BI')
print('=' * 55)

dashboards = [
    ('Dashboard 1 - Visao Executiva',
     ['KPI Card: Win Rate Global (49.9%)',
      'KPI Card: Total de Partidas (9.879)',
      'KPI Card: Gold Diff Medio em Vitorias (+1.543)',
      'Donut: Distribuicao Vitorias vs Derrotas',
      'Bar Chart: Win Rate por Categoria de Gold',
      'Slicer: first_blood_label | dragon_label | categoria_gold']),
    ('Dashboard 2 - Early Game',
     ['Clustered Bar: Media de Kills, Dragons, Wards por Resultado',
      'Line: Gold Diff vs Win Rate por faixa',
      'Matrix: First Blood x Dragon - Taxa de Vitoria',
      'Scatter: Gold Diff (X) x Experiencia Diff (Y), cor=resultado',
      'KPI: Impacto do First Blood em p.p.']),
    ('Dashboard 3 - Perfis Estrategicos (Clusters)',
     ['Bar Chart: Win Rate por Cluster (C0 a C3)',
      'Donut: Distribuicao de Partidas por Cluster',
      'Table: Perfil medio de cada cluster',
      '100% Stacked Bar: Vitorias dentro de cada cluster']),
    ('Dashboard 4 - Modelo de Machine Learning',
     ['KPI Card: AUC ROC (81.06%)',
      'KPI Card: Acuracia (73.38%)',
      'KPI Card: F1-Score (73.03%)',
      'Table: Ranking dos 7 modelos',
      'Bar: CV AUC dos 7 modelos',
      'Text: Paragrafo executivo com insights']),
]

for nome, visuais in dashboards:
    print()
    print(nome)
    for v in visuais:
        print('  -', v)

print()
print('PARAGRAFO EXECUTIVO FINAL')
print('-' * 55)
print('O modelo de Regressao Logistica, treinado com dados dos')
print('primeiros 10 minutos de 7.903 partidas ranqueadas,')
print('alcancou AUC ROC de 81.06% e acuracia de 73.38%,')
print('superando Random Forest, XGBoost e SVM em validacao cruzada.')
print()
print('Principais drivers de vitoria:')
print('  1. blueGoldDiff (+0.530) - maior impacto isolado')
print('  2. blueTotalGold (+0.418) - volume economico total')
print('  3. redTotalGold (-0.444) - ouro do adversario')
print()
print('Recomendacoes:')
print('  1. Maximizar ouro nos primeiros 10 minutos')
print('  2. Conquistar o Dragao - desempata partidas equilibradas')
print('  3. Buscar o First Blood - eleva win rate em +20.6 p.p.')
print()
print('FASE 7 CONCLUIDA!')
print('PROJETO COMPLETO - 7 FASES FINALIZADAS!')

FASE 7 - Camada de Negocios e Business Intelligence

KPI CARDS PARA O POWER BI
----------------------------------------
Total de Partidas          : 9879
Win Rate Azul Global       : 49.9 %
Gold Diff Medio (Vitorias) : +1271.0
Gold Diff Medio (Derrotas) : -1237.0
Win Rate COM First Blood   : 59.9 %
Win Rate SEM First Blood   : 39.7 %
Impacto do First Blood     : +20.2 p.p.
Win Rate COM Dragao        : 64.1 %
Win Rate SEM Dragao        : 41.9 %
Impacto do Dragao          : +22.2 p.p.

WIN RATE POR CATEGORIA DE GOLD:
  Forte Vantagem: 83.8%
  Leve Vantagem: 59.4%
  Leve Desvantagem: 39.8%
  Forte Desvantagem: 16.5%

WIN RATE POR DRAGON LABEL:
  0 Dragoes: 41.9%
  1 Dragao: 64.1%

FORMULAS DAX PARA O POWER BI

DAX 1. Taxa de Vitoria Azul (medida base)

Taxa de Vitoria Azul =
DIVIDE(
    CALCULATE(COUNTROWS(lol_matches), lol_matches[blueWins] = 1),
    COUNTROWS(lol_matches),
    0
)

DAX 2. Taxa de Vitoria com First Blood

Vitoria com First Blood =
CALCULATE(
    DIVIDE(
        CALCULATE

In [11]:
# CELULA 11 - Criar README, gitignore, requirements e ZIP
import zipfile

print('Gerando arquivos finais...')

readme_content = open('/dev/stdin').read() if False else """# League of Legends - Previsao de Vitoria em Ranked

[![Python](https://img.shields.io/badge/Python-3.10+-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Scikit-learn](https://img.shields.io/badge/Scikit--learn-1.5-F7931E?style=for-the-badge&logo=scikit-learn&logoColor=white)](https://scikit-learn.org)
[![XGBoost](https://img.shields.io/badge/XGBoost-2.0-189FDD?style=for-the-badge)](https://xgboost.readthedocs.io)
[![SQLite](https://img.shields.io/badge/SQLite-3-003B57?style=for-the-badge&logo=sqlite&logoColor=white)](https://sqlite.org)
[![Power BI](https://img.shields.io/badge/Power%20BI-F2C811?style=for-the-badge&logo=powerbi&logoColor=black)](https://powerbi.microsoft.com)
[![Jupyter](https://img.shields.io/badge/Jupyter-Notebook-F37626?style=for-the-badge&logo=jupyter&logoColor=white)](https://jupyter.org)

> **Projeto Final - Curso de Ciencia de Dados**

---

## Sobre o Projeto

**Problema de Negocio:** Prever se o time azul vencera (blueWins) com base nos primeiros 10 minutos de partidas ranqueadas.

| | |
|---|---|
| Tipo de Problema | Classificacao Binaria Supervisionada |
| Dataset | 9.879 partidas x 40 atributos |
| Target | blueWins - balanceado (49.9% vs 50.1%) |
| Modelo Final | Regressao Logistica |
| AUC ROC | 81.06% |
| Acuracia | 73.38% |

---

## Resultados em Destaque

| Metrica | Valor |
|---------|-------|
| CV AUC (5-Fold) | 80.87% |
| AUC ROC Teste | 81.06% |
| Acuracia | 73.38% |
| F1-Score | 73.03% |

---

## Top 5 Insights de Negocio

| # | Insight | Impacto |
|---|---------|---------|
| 1 | Gold Diff e o maior preditor | r = 0.63 com blueWins |
| 2 | First Blood eleva a win rate | 40% -> 60.6% (+20.6 p.p.) |
| 3 | Dragao desempata partidas equilibradas | 38.6% -> 60.5% (+21.9 p.p.) |
| 4 | Q4 de Gold Diff quase garante vitoria | 84.6% de win rate |
| 5 | Problema linearmente separavel | LogReg supera RF, XGB e SVM |

---

## Visualizacoes

### EDA e Storytelling
| Visao Geral | O Ouro Decide |
|:-----------:|:-------------:|
| ![g01](outputs/figures/grafico_01_visao_geral.png) | ![g02](outputs/figures/grafico_02_ouro_decide.png) |

| Early Game | Perfil Vencedor |
|:----------:|:---------------:|
| ![g03](outputs/figures/grafico_03_early_game.png) | ![g04](outputs/figures/grafico_04_perfil_vencedor.png) |

### Pre-Processamento e PCA
| Multicolinearidade | Variancia PCA |
|:-----------------:|:-------------:|
| ![g05](outputs/figures/grafico_05_multicolinearidade_vif.png) | ![g06](outputs/figures/grafico_06_pca_variancia.png) |

### K-Means
| Elbow + Silhouette | Perfil dos Clusters |
|:------------------:|:-------------------:|
| ![g07](outputs/figures/grafico_07_elbow_silhouette.png) | ![g08](outputs/figures/grafico_08_cluster_perfil.png) |

![g09](outputs/figures/grafico_09_silhouette_pca_scatter.png)

### Batalha dos Modelos
![g10](outputs/figures/grafico_10_batalha_modelos.png)

| Duelos | ROC + Ranking |
|:------:|:-------------:|
| ![g11](outputs/figures/grafico_11_duelos_roc.png) | ![g12](outputs/figures/grafico_12_roc_ranking_tabela.png) |

### Avaliacao Final
| Matriz de Confusao + ROC | Feature Importance |
|:------------------------:|:-----------------:|
| ![g13](outputs/figures/grafico_13_confusao_report_roc.png) | ![g14](outputs/figures/grafico_14_feature_importance.png) |

---

## Estrutura do Projeto

```
lol-ranked-prediction/
|
|-- notebooks/
|   |-- fase_01_setup_sql.ipynb
|   |-- fase_02_eda_storytelling.ipynb
|   |-- fase_03_preprocessing_pipeline.ipynb
|   |-- fase_04_kmeans.ipynb
|   |-- fase_05_modelagem.ipynb
|   |-- fase_06_avaliacao.ipynb
|   +-- fase_07_bi_dax.ipynb
|
|-- outputs/
|   |-- figures/    <- 14 graficos
|   +-- data/
|       |-- dataset_tratado_powerbi.csv
|       |-- cluster_profile.csv
|       +-- model_results.csv
|
|-- requirements.txt
|-- .gitignore
+-- README.md
```

---

## Como Executar

```bash
git clone https://github.com/BrunoMarino2001/lol-ranked-prediction.git
cd lol-ranked-prediction
pip install -r requirements.txt
```

---

## Stack Tecnologica

| Categoria | Tecnologia |
|-----------|------------|
| Linguagem | Python 3.10+ |
| Banco de Dados | SQLite3 |
| Manipulacao | Pandas, NumPy |
| Visualizacao | Matplotlib, Seaborn, SciPy |
| Pre-processamento | Scikit-learn (Pipeline, StandardScaler, PCA) |
| Clusterizacao | KMeans, silhouette_score |
| Modelos ML | GaussianNB, LogisticRegression, DecisionTree, RandomForest, SVM, XGBoost |
| Otimizacao | GridSearchCV, StratifiedKFold |
| BI | Power BI + DAX |
| Ambiente | Jupyter Notebook / Google Colab |

---

## Roadmap

```
Fase 1 - Setup, SQLite e Ingestao de Dados
Fase 2 - EDA e Storytelling (14 visualizacoes)
Fase 3 - Pre-Processamento + Checkpoint BI
Fase 4 - K-Means: 4 perfis de partida
Fase 5 - 6 modelos + GridSearchCV
Fase 6 - Avaliacao + Feature Importance
Fase 7 - Plano de BI + 7 Formulas DAX
```

---

*Projeto finalizado | Modelo: Regressao Logistica | AUC ROC: 81.06%*
"""

gitignore_content = """# Banco de dados
*.db

# Dados brutos e binarios
Base_M43_Pratique_LOL_RANKED_WIN.csv
*.parquet
*.npy

# Modelos serializados
outputs/models/

# Cache Python
__pycache__/
*.pyc
*.pyo
.ipynb_checkpoints/
*/.ipynb_checkpoints/

# Ambiente virtual
venv/
.env
.venv/

# Sistema operacional
.DS_Store
Thumbs.db
"""

requirements_content = """pandas==2.2.2
numpy==1.26.4
pyarrow==16.1.0
matplotlib==3.9.0
seaborn==0.13.2
scipy==1.13.1
scikit-learn==1.5.0
xgboost==2.0.3
joblib==1.4.2
jupyter==1.0.0
ipykernel==6.29.4
"""

readme_path = os.path.join(BASE_DIR, 'README.md')
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)

gitignore_path = os.path.join(BASE_DIR, '.gitignore')
with open(gitignore_path, 'w') as f:
    f.write(gitignore_content)

req_path = os.path.join(BASE_DIR, 'requirements.txt')
with open(req_path, 'w') as f:
    f.write(requirements_content)

print('Arquivos gerados: README.md, .gitignore, requirements.txt')

ZIP_PATH = '/content/lol_ranked_prediction_COMPLETO.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(DIRS['figures']):
        zf.write(os.path.join(DIRS['figures'], f), os.path.join('outputs','figures',f))
    for f in ['dataset_tratado_powerbi.csv','cluster_profile.csv','model_results.csv']:
        fp = os.path.join(DIRS['data'], f)
        if os.path.exists(fp):
            zf.write(fp, os.path.join('outputs','data',f))
    zf.write(readme_path, 'README.md')
    zf.write(gitignore_path, '.gitignore')
    zf.write(req_path, 'requirements.txt')

print('ZIP criado:', round(os.path.getsize(ZIP_PATH)/1024/1024, 2), 'MB')
print('Baixando...')
from google.colab import files
files.download(ZIP_PATH)
print('Download iniciado!')

Gerando arquivos finais...
Arquivos gerados: README.md, .gitignore, requirements.txt
ZIP criado: 2.51 MB
Baixando...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download iniciado!
